In [2]:
# Librerias
import pandas as pd
from sklearn.model_selection import train_test_split
import os
import numpy as np

In [3]:
# =================
# Cargar dataset
# =================

ruta_csv = os.path.join("..", "data", "processed","ESdE_2023_renombrado.csv" )
df_adulto = pd.read_csv(ruta_csv, sep=",", encoding="latin-1")

print("Dataset cargado")

Dataset cargado


In [4]:
# HIPERTENSIÓN
df_hta = df_adulto.copy()

condicion_valida_hta = (
    df_hta["Hipertension_Autorreporte"].isin([1.0, 2.0, 1, 2]) | df_hta["Hipertension_Autorreporte"].isna()
) & (
    df_hta["Hipertension_Diag_Medico"].isin([1.0, 2.0, 1, 2]) | df_hta["Hipertension_Diag_Medico"].isna()
)

df_hta = df_hta[condicion_valida_hta].copy()

df_hta["Target_HTA"] = (df_hta["Hipertension_Diag_Medico"] == 1).astype(int)

# DIABETES 

df_diab = df_adulto.copy()

condicion_valida_diab = (
    df_diab["Diabetes_Autorreporte"].isin([1.0, 2.0, 1, 2]) | df_diab["Diabetes_Autorreporte"].isna()
) & (
    df_diab["Diabetes_Diag_Medico"].isin([1.0, 2.0, 1, 2]) | df_diab["Diabetes_Diag_Medico"].isna()
)

df_diab = df_diab[condicion_valida_diab].copy()

df_diab["Target_Diab"] = (df_diab["Diabetes_Diag_Medico"] == 1).astype(int)


print('=== HIPERTENSIÓN ===')
print(f'Total de registros limpios: {len(df_hta)}')
print(df_hta['Target_HTA'].value_counts())

print('\n=== DIABETES ===')
print(f'Total de registros limpios: {len(df_diab)}')
print(df_diab['Target_Diab'].value_counts())

=== HIPERTENSIÓN ===
Total de registros limpios: 20693
Target_HTA
0    14919
1     5774
Name: count, dtype: int64

=== DIABETES ===
Total de registros limpios: 20869
Target_Diab
0    18864
1     2005
Name: count, dtype: int64


In [5]:
# ===========================================
# PREPARACIÓN PARA EL MODELO DE HIPERTENSIÓN
# ===========================================

# Target: Hipertensión
y_hta =df_hta["Target_HTA"]

# Separamos la variable objetivo
X_hta = df_hta.drop(columns=["Target_HTA"])

#Split estratificado 
X_train_hta, X_test_hta, y_train_hta, y_test_hta = train_test_split(
    X_hta,
    y_hta,
    test_size=0.20,
    random_state=42,
    stratify=y_hta
)

# =========================================
# PREPARACIÓN PARA EL MODELO DE DIABETES
# ========================================

# Target: Diabetes
y_diab = df_diab["Target_Diab"]

# Separamos la variable objetivo
X_diab =df_diab.drop(columns=["Target_Diab"])

# Split estratificado
X_train_diab, X_test_diab, y_train_diab, y_test_diab = train_test_split(
    X_diab,
    y_diab,
    test_size=0.20,
    random_state=42,
    stratify=y_diab
)

print("Divisiones Train y Test creadas con éxito para ambas enfermedades")

Divisiones Train y Test creadas con éxito para ambas enfermedades


# CONSOLIDAR VARIABLES DE TABACO

FUMADOR DIARIO ACTUAL

Si -> "Tabaquismo" == 1; and "Duracion_Cons_Tabaco" < 5 años 
    entonces "Fumador_Corto_Plazo"

Si -> "Tabaquismo" == 1; and "Duracion_Cons_Tabaco" > 5 años and < 15 años
    entonces "Fumador_Mediano_Plazo"

Si -> "Tabaquismo" == 1; and "Duracion_Cons_Tabaco" > 15 años
    entonces "Fumador_Largo_Plazo"

FUMADOR OCASIONAL ACTUAL

Si -> "Tabaquismo" == 2; and "Ex_Fumador" == 1; and "Duracion_Cons_Tabaco" < 5 años 
    entonces "Fumador_Ocasional_Bajo_Historial"

Si -> "Tabaquismo" == 2; and "Ex_Fumador" == 1; and "Duracion_Cons_Tabaco"> 5 años and < 15 años
    entonces "Fumador_Ocasional_Mediano_Historial"

Si -> "Tabaquismo" == 2; and "Ex_Fumador" == 1; and "Duracion_Cons_Tabaco" > 15 años
    entonces "Fumador_Ocasional_Alto_Historial"

Si -> "Tabaquismo" == 2; and "Ex_Fumador" == 2; 
    entonces "Fumador_Ocasional"

Si -> "Tabaquismo" == 2; and "Ex_Fumador" == 9;
    entonces "Fumador_Ocasional"

EX-FUMADOR

Si -> "Tabaquismo" == 3; and "Ex_Fumador" == 1; and "Duracion_Cons_Tabaco" < 5 años 
    entonces "Fumador_Corto_Plazo"

Si -> "Tabaquismo" == 3; and "Ex_Fumador" == 1; and "Duracion_Cons_Tabaco" > 5 años and < 15 años
    entonces "Fumador_Mediano_Plazo"

Si -> "Tabaquismo" == 3; and "Ex_Fumador" == 1; and "Duracion_Cons_Tabaco" > 15 años
    entonces "Fumador_Largo_Plazo"

Si -> "Tabaquismo" == 3; and "Ex_Fumador" == 2; 
    entonces "Fumador_Ocasional"

Si -> "Tabaquismo" == 3; and "Ex_Fumador" == 9; 
    entonces "Fumador_Ocasional"

NUNCA FUMADOR

Si -> "Tabaquismo" == 4;
    entonces "No_Fumador"

NO CONTESTA

Si -> "Tabaquismo" == 9;
    entonces "No_Contesta"

In [6]:
# ===================================================
# FUNCIÓN QUE CONSOLIDA LAS TRES VARIABLES DE TABACO
# ===================================================

def consolidar_tabaco(row):
    tab = row.get("Tabaquismo")
    ex = row.get("Ex_Fumador")
    dur = row.get("Duracion_Cons_Tabaco")
    
    # No sabe / No contesta en la pregunta principal
    if tab == 9 or pd.isna(tab):
        return "No_Contesta"
    
    # Nunca fumador
    if tab == 4:
        return "No_Fumador"
    
    # Gestionar nulos y códigos especiales (99) en la duración
    es_dur_valida = pd.isna(dur) or dur not in [99, 99.0]
    dur_num = float(dur) if es_dur_valida else 0.0
    
    # Fumador diario actual
    if tab == 1:
        if es_dur_valida or dur_num < 5:
            return "Fumador_Corto_Plazo"
        elif 5 <= dur_num <= 15:
            return "Fumador_Mediano_Plazo"
        else:
            return "Fumador_Largo_Plazo"
        
    # Fumador ocasional actual
    if tab == 2:
        if ex == 1 and es_dur_valida:
            if dur_num < 5:
                return "Fumador_Ocasional_Bajo_Historial"
            elif  5 <= dur_num <= 15:
                return "Fumador_Ocasional_Medio_Historial"
            else:
                return "Fumador_Ocasional_Alto_Historial"
        else:
            return "Fumador_Ocasional"
        
    # Ex-Fumador
    if tab == 3:
        if ex ==1:
            if es_dur_valida or dur_num < 5:
                return "Ex_Fumador_Corto_Plazo"
            elif 5 <= dur_num <= 15:
                return "Ex_Fumador_Mediano_Plazo"
            else: 
                return "Ex_Fumador_Largo_Plazo"
        else:
            return "Ex_Fumador_Ocasional"
    
    return "No_Fumador"
    

In [7]:
# ==========================================
# AGREGAMOS LA FUNCIÓN DE TABACO A LOS DATOS
# ==========================================


# Aplicar función al conjunto de Hipertensión
X_train_hta["Tabaco_Consolidado"] = X_train_hta.apply(consolidar_tabaco, axis=1)
X_test_hta["Tabaco_Consolidado"] = X_test_hta.apply(consolidar_tabaco, axis=1)

# Aplicar función al conjunto de Diabetes
X_train_diab["Tabaco_Consolidado"] = X_train_diab.apply(consolidar_tabaco, axis=1)
X_test_diab["Tabaco_Consolidado"] = X_test_diab.apply(consolidar_tabaco, axis=1)

# SELECCIÓN DE VARIABLES PARA EL MODELO

In [8]:
# =================================
# Lista con variables segun target
# =================================

# Lista con variables para hipertensión
variables_hta = [
    "Sexo", "Edad", "IMC", "Nivel_Estudios", "Clase_Social", "Act_Fisica",
    "Hs_Sentado", "Tabaco_Consolidado", "Cons_Alcohol",
    "Cons_Fruta", "Cons_Carne", "Cons_Huevo", "Cons_Zumos_Naturales",
    "Cons_Pescado", "Cons_Carbohidrato", "Cons_Cereales", "Cons_Verduras",
    "Cons_Legumbres", "Cons_Embutidos", "Cons_Lacteos", "Cons_Dulces",
    "Cons_Refrescos", "Cons_Comida_Rapida", "Cons_Snacks_Salados"
]

# Lista con variables para diabetes
variables_diab = [
    "Sexo", "Edad", "IMC", "Nivel_Estudios", "Clase_Social", "Act_Fisica",
    "Hs_Sentado", "Tabaco_Consolidado", "Cons_Alcohol",
    "Cons_Fruta", "Cons_Carne", "Cons_Huevo", "Cons_Zumos_Naturales",
    "Cons_Pescado", "Cons_Carbohidrato", "Cons_Cereales", "Cons_Verduras",
    "Cons_Legumbres", "Cons_Embutidos", "Cons_Lacteos", "Cons_Dulces",
    "Cons_Refrescos", "Cons_Comida_Rapida", "Cons_Snacks_Salados"
]

# ========================================
# Asignar lista de variables a los target
# ========================================

X_train_hta = X_train_hta[variables_hta]
X_test_hta = X_test_hta[variables_hta]

X_train_diab = X_train_diab[variables_diab]
X_test_diab = X_test_diab[variables_diab]

# TRATAMIENTO DE VALORES NULOS Y CÓDIGOS 9 / 99 DE LAS VARIABLES PREDICTORAS

In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Definir los grupos exactos 
var_numerica = ["Hs_Sentado"] # Tratar '99'como nulo y luego aplicar la Mediana

var_ordinal = ["IMC"] # Tratar '9' como nulo y luego 'No consta' -> OHE

vars_frecuencias = ["Cons_Fruta", "Cons_Carne", "Cons_Huevo",
    "Cons_Pescado", "Cons_Carbohidrato", "Cons_Cereales", "Cons_Verduras",
    "Cons_Legumbres", "Cons_Embutidos", "Cons_Lacteos", "Cons_Dulces",
    "Cons_Refrescos", "Cons_Comida_Rapida", "Cons_Snacks_Salados",
    "Cons_Zumos_Naturales"] # Tratar '9' como nulo e imputar por la moda -> OHE

# Variables categóricas con '9' o '99' a tratar como nan y luego 'No consta' -> OHE
vars_cat_con_9 = ["Clase_Social", "Act_Fisica", ] 

vars_cat_con_99 = ["Nivel_Estudios", "Cons_Alcohol"]

# Variables categóricas ya limpias, solo falta OHE 
vars_cat_limpias =["Sexo", "Tabaco_Consolidado"]

# Variables numericas a escalar
vars_numericas = ["Hs_Sentado", "Edad"]

# Lista total de variables categóricas para el One Hot Encoder
vars_categoricas = vars_cat_con_9 + vars_cat_con_99 + vars_cat_limpias


In [12]:
# Limpieza de variables con 9
for col in (var_ordinal + vars_frecuencias + vars_cat_con_9):
    X_train_hta[col] = X_train_hta[col].replace([9, 9.0], np.nan)
    X_test_hta[col] = X_test_hta[col].replace([9, 9.0], np.nan)
    X_train_diab[col] = X_train_diab[col].replace([9, 9.0], np.nan)
    X_test_diab[col] = X_test_diab[col].replace([9, 9.0], np.nan)
    
# Limpieza de variables con 99    
for col in var_numerica + vars_cat_con_99:
    X_train_hta[col] = X_train_hta[col].replace([99, 99.0], np.nan)
    X_test_hta[col] = X_test_hta[col].replace([99, 99.0], np.nan)
    X_train_diab[col] = X_train_diab[col].replace([99, 99.0], np.nan)
    X_test_diab[col] = X_test_diab[col].replace([99, 99.0], np.nan)
    
print("Limpieza de valores 9 y 99 completada")


Limpieza de valores 9 y 99 completada


In [13]:
# Imputar por la Moda variables de frecuencia con nulos

# IMPUTAR POR MODA
imputar_x_moda = SimpleImputer(strategy="most_frequent")

# IMPUTAR HIPERTENSIÓN 
X_train_hta[vars_frecuencias] = pd.DataFrame(
    imputar_x_moda.fit_transform(X_train_hta[vars_frecuencias]),
    columns=vars_frecuencias,
    index=X_train_hta.index)

X_test_hta[vars_frecuencias] = pd.DataFrame(
    imputar_x_moda.fit_transform(X_test_hta[vars_frecuencias]),
    columns=vars_frecuencias,
    index=X_test_hta.index)

# IMPUTAR DIABETES 
imputar_x_moda_diab = SimpleImputer(strategy="most_frequent")

X_train_diab[vars_frecuencias] = pd.DataFrame(
    imputar_x_moda_diab.fit_transform(X_train_diab[vars_frecuencias]),
    columns=vars_frecuencias,
    index=X_train_diab.index)

X_test_diab[vars_frecuencias] = pd.DataFrame(
    imputar_x_moda_diab.fit_transform(X_test_diab[vars_frecuencias]),
    columns=vars_frecuencias,
    index=X_test_diab.index)

print("Imputación por MODA completada con éxito")

Imputación por MODA completada con éxito


In [14]:
# Comprobar que no quedan nulos en las variables de frecuencia/alimentos
nulos_hta_train = X_train_hta[vars_frecuencias].isna().sum().sum()
nulos_hta_test = X_test_hta[vars_frecuencias].isna().sum().sum()

nulos_diab_train = X_train_diab[vars_frecuencias].isna().sum().sum()
nulos_diab_test = X_test_diab[vars_frecuencias].isna().sum().sum()

print(f"Nulos restantes en HTA Train: {nulos_hta_train}")
print(f"Nulos restantes en HTA Test:  {nulos_hta_test}")
print(f"Nulos restantes en Diabetes Train: {nulos_diab_train}")
print(f"Nulos restantes en Diabetes Test:  {nulos_diab_test}")

Nulos restantes en HTA Train: 0
Nulos restantes en HTA Test:  0
Nulos restantes en Diabetes Train: 0
Nulos restantes en Diabetes Test:  0


In [15]:
# Imputar nulos por la mediana 
imputar_x_mediana = SimpleImputer(strategy="median")

# IMPUTAR HIPERTENSIÓN 
X_train_hta[var_numerica] = pd.DataFrame(
    imputar_x_mediana.fit_transform(X_train_hta[var_numerica]),
    columns=var_numerica,
    index=X_train_hta.index)

X_test_hta[var_numerica] = pd.DataFrame(
    imputar_x_mediana.fit_transform(X_test_hta[var_numerica]),
    columns=var_numerica,
    index=X_test_hta.index)

# IMPUTAR DIABETES 
imputar_x_mediana_diab = SimpleImputer(strategy="median")

X_train_diab[var_numerica] = pd.DataFrame(
    imputar_x_mediana_diab.fit_transform(X_train_diab[var_numerica]),
    columns=var_numerica,
    index=X_train_diab.index)

X_test_diab[var_numerica] = pd.DataFrame(
    imputar_x_mediana_diab.fit_transform(X_test_diab[var_numerica]),
    columns=var_numerica,
    index=X_test_diab.index)

print("Imputación por MEDIANA completada con éxito")

Imputación por MEDIANA completada con éxito


In [16]:
# 1. Verificar nulos restantes
nulos_mediana_hta = X_train_hta[var_numerica].isna().sum().sum()
nulos_mediana_diab = X_train_diab[var_numerica].isna().sum().sum()

print(f'Nulos restantes en HTA (Mediana): {nulos_mediana_hta}')
print(f'Nulos restantes en Diabetes (Mediana): {nulos_mediana_diab}')

# 2. Ver las medianas aprendidas por el modelo
df_medianas = pd.DataFrame(
    {
        'Variable': var_numerica,
        'Mediana_HTA': imputar_x_mediana.statistics_,
        'Mediana_Diabetes': imputar_x_mediana_diab.statistics_,
    }
)

print('\n=== MEDIANAS CALCULADAS E IMPUTADAS ===')
print(df_medianas)

Nulos restantes en HTA (Mediana): 0
Nulos restantes en Diabetes (Mediana): 0

=== MEDIANAS CALCULADAS E IMPUTADAS ===
     Variable  Mediana_HTA  Mediana_Diabetes
0  Hs_Sentado          5.0               5.0


In [17]:
# Pasamos nulos a "No_Contesta"

vars_categoricas_a_tratar = vars_cat_con_9 + vars_cat_con_99 + var_ordinal

for col in vars_categoricas_a_tratar:
    # HIPERTENSIÓN
    X_train_hta[col] = X_train_hta[col].fillna("No_Contesta").astype(str)
    X_test_hta[col] = X_test_hta[col].fillna("No_Contesta").astype(str)
    # DIABETES
    X_train_diab[col] = X_train_diab[col].fillna("No_Contesta").astype(str)
    X_test_diab[col] = X_test_diab[col].fillna("No_Contesta").astype(str)
    
    
print("Relleno de nulos categóricos con 'No_Contesta' completado con éxito")

Relleno de nulos categóricos con 'No_Contesta' completado con éxito


In [18]:
nulos_cat_hta = X_train_hta[vars_categoricas_a_tratar].isna().sum().sum()
nulos_cat_diab = X_train_diab[vars_categoricas_a_tratar].isna().sum().sum()

print(f"Nulos en categóricas HTA Train: {nulos_cat_hta}")
print(f"Nulos en categóricas Diabetes Train: {nulos_cat_diab}")

Nulos en categóricas HTA Train: 0
Nulos en categóricas Diabetes Train: 0


# ONE HOT ENCODER

In [19]:
# GRUPO ESCALADO 
vars_a_escalar = (vars_frecuencias + vars_numericas)

#Grupo one-hot-encoder 
vars_a_ohe = (var_ordinal + vars_categoricas)

# PREPROCESADOR 
preprocesador = ColumnTransformer(
    transformers=[
        ("scaler", StandardScaler(), vars_a_escalar),
        ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False), vars_a_ohe)])

# APLICAR A HIPERTENSIÓN 
X_train_hta_prep = preprocesador.fit_transform(X_train_hta)
X_test_hta_prep = preprocesador.transform(X_test_hta)

# APLICAR A DIABETES 
X_train_diab_prep = preprocesador.fit_transform(X_train_diab)
X_test_diab_prep = preprocesador.transform(X_test_diab)

print("")

In [20]:
print(
    f"Filas/Columnas originales HTA Train: {X_train_hta.shape}"
)  # Ej. (10000, 20)
print(
    f"Filas/Columnas procesadas HTA Train: {X_train_hta_prep.shape}"
)  # Ej. (10000, 45)

print(
    f"\nFilas/Columnas originales Diabetes Train: {X_train_diab.shape}"
)
print(
    f"Filas/Columnas procesadas Diabetes Train: {X_train_diab_prep.shape}"
)

Filas/Columnas originales HTA Train: (16554, 24)
Filas/Columnas procesadas HTA Train: (16554, 63)

Filas/Columnas originales Diabetes Train: (16695, 24)
Filas/Columnas procesadas Diabetes Train: (16695, 63)


In [21]:
# Obtener los nombres de las columnas transformadas
nombres_columnas_hta = preprocesador.get_feature_names_out()

print("=== NOMBRES DE LAS NUEVAS COLUMNAS GENERADAS (HTA) ===")
for col in nombres_columnas_hta:
    print(col)

=== NOMBRES DE LAS NUEVAS COLUMNAS GENERADAS (HTA) ===
scaler__Cons_Fruta
scaler__Cons_Carne
scaler__Cons_Huevo
scaler__Cons_Pescado
scaler__Cons_Carbohidrato
scaler__Cons_Cereales
scaler__Cons_Verduras
scaler__Cons_Legumbres
scaler__Cons_Embutidos
scaler__Cons_Lacteos
scaler__Cons_Dulces
scaler__Cons_Refrescos
scaler__Cons_Comida_Rapida
scaler__Cons_Snacks_Salados
scaler__Cons_Zumos_Naturales
scaler__Hs_Sentado
scaler__Edad
ohe__IMC_1.0
ohe__IMC_2.0
ohe__IMC_3.0
ohe__IMC_4.0
ohe__IMC_No_Contesta
ohe__Clase_Social_1.0
ohe__Clase_Social_2.0
ohe__Clase_Social_3.0
ohe__Clase_Social_4.0
ohe__Clase_Social_5.0
ohe__Clase_Social_6.0
ohe__Clase_Social_No_Contesta
ohe__Act_Fisica_1.0
ohe__Act_Fisica_2.0
ohe__Act_Fisica_3.0
ohe__Act_Fisica_4.0
ohe__Act_Fisica_No_Contesta
ohe__Nivel_Estudios_2.0
ohe__Nivel_Estudios_4.0
ohe__Nivel_Estudios_5.0
ohe__Nivel_Estudios_6.0
ohe__Nivel_Estudios_7.0
ohe__Nivel_Estudios_8.0
ohe__Nivel_Estudios_9.0
ohe__Nivel_Estudios_No_Contesta
ohe__Cons_Alcohol_1.0
ohe__C

In [22]:
import pandas as pd

# Convertir la matriz numpy procesada a DataFrame para inspección visual
df_hta_revision = pd.DataFrame(X_train_hta_prep, columns=nombres_columnas_hta)

# Ver las primeras 5 filas
df_hta_revision.head()

,scaler__Cons_Fruta,scaler__Cons_Carne,scaler__Cons_Huevo,scaler__Cons_Pescado,scaler__Cons_Carbohidrato,scaler__Cons_Cereales,scaler__Cons_Verduras,scaler__Cons_Legumbres,scaler__Cons_Embutidos,scaler__Cons_Lacteos,...,ohe__Sexo_2,ohe__Tabaco_Consolidado_Ex_Fumador_Corto_Plazo,ohe__Tabaco_Consolidado_Ex_Fumador_Ocasional,ohe__Tabaco_Consolidado_Fumador_Corto_Plazo,ohe__Tabaco_Consolidado_Fumador_Ocasional,ohe__Tabaco_Consolidado_Fumador_Ocasional_Alto_Historial,ohe__Tabaco_Consolidado_Fumador_Ocasional_Bajo_Historial,ohe__Tabaco_Consolidado_Fumador_Ocasional_Medio_Historial,ohe__Tabaco_Consolidado_No_Contesta,ohe__Tabaco_Consolidado_No_Fumador
0,-0.715658,1.094280,0.835535,-0.751428,0.998031,-0.586830,-1.064784,0.452944,-1.235941,0.326103,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,-0.715658,-1.760379,0.835535,-0.751428,-0.969974,1.970710,-1.064784,-0.669181,0.274730,-0.548967,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,0.088674,0.142727,-0.205284,1.394177,0.014028,0.265683,-1.064784,0.452944,1.030065,-0.548967,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.893005,-0.808826,0.835535,-0.751428,0.014028,1.970710,0.688741,0.452944,-0.480606,1.201174,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,-0.715658,0.142727,1.876353,0.321374,-0.969974,-0.586830,-1.064784,-0.669181,1.785400,-0.548967,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [23]:
print(
    "Nulos en HTA Train procesado:",
    pd.DataFrame(X_train_hta_prep).isna().sum().sum(),
)
print(
    "Nulos en Diabetes Train procesado:",
    pd.DataFrame(X_train_diab_prep).isna().sum().sum(),
)

Nulos en HTA Train procesado: 0
Nulos en Diabetes Train procesado: 0


# GUARDAR LOS DATOS PREPROCESADOS 

In [24]:
# ===================================================
# Se recuperan los nombres de las columnas generadas 
# ===================================================

nombres_cols = preprocesador.get_feature_names_out()

# ==============================================
# Se convierten las matrices Numpy a Dataframes 
# ==============================================

# HIPERTENSIÓN 
X_train_hta_df = pd.DataFrame(X_train_hta_prep, columns=nombres_cols)
X_test_hta_df = pd.DataFrame(X_test_hta_prep, columns=nombres_cols)

# DIABETES 
X_train_diab_df = pd.DataFrame(X_train_diab_prep, columns=nombres_cols)
X_test_diab_df = pd.DataFrame(X_test_diab_prep, columns=nombres_cols)

# =====================================================
# Guardar los archivos procesados en la carpeta 'data'
# =====================================================

# HIPERTENSIÓN 
X_train_hta_df.to_csv("../data/processed/X_train_hta_df.csv", index=False)
X_test_hta_df.to_csv("../data/processed/X_test_hta_df.csv", index=False)
y_train_hta.to_csv("../data/processed/y_train_hta.csv", index=False)
y_test_hta.to_csv("../data/processed/y_test_hta.csv", index=False)

# DIABETES
X_train_diab_df.to_csv("../data/processed/X_train_diab_df.csv", index=False)
X_test_diab_df.to_csv("../data/processed/X_test_diab_df.csv", index=False)
y_train_diab.to_csv("../data/processed/y_train_diab.csv", index=False)
y_test_diab.to_csv("../data/processed/y_test_diab.csv", index=False)

print("ARCHIVOS GUARDADOS CON ÉXITO EN LA CARPETA DATA")

ARCHIVOS GUARDADOS CON ÉXITO EN LA CARPETA DATA


# GUARDAR PESOS MUESTRALES (FACTOR ADULTO) PARA EL MODELADO

In [25]:
# Se selecciona la variable desde el dataset original de acuerdo con los índices 
# de los conjuntos de train y test, garantizando que los pesos muestrales 
# correspondan a cada observación.

sample_weight_train_hta = df_adulto.loc[X_train_hta.index, "FACTORADULTO"]
sample_weight_test_hta = df_adulto.loc[X_test_hta.index, "FACTORADULTO"]

sample_weight_train_diab = df_adulto.loc[X_train_diab.index, "FACTORADULTO"]
sample_weight_test_diab = df_adulto.loc[X_test_diab.index, "FACTORADULTO"]

# Se guardan los pesos en data/processed
sample_weight_train_hta.to_csv("../data/processed/sample_weight_train_hta.csv", index=False)
sample_weight_test_hta.to_csv("../data/processed/sample_weight_test_hta.csv", index=False)
sample_weight_train_diab.to_csv("../data/processed/sample_weight_train_diab.csv", index=False)
sample_weight_test_diab.to_csv("../data/processed/sample_weight_test_diab.csv", index=False)

print("¡FACTORADULTO extraído y guardado perfectamente!")

¡FACTORADULTO extraído y guardado perfectamente!
